In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor


In [3]:
#Load Data
df = pd.read_csv("E:/Inventory_managment/data/retail_store_inventory_feature_engineered.csv")
print(df.head())

   Inventory Level  Price  Discount  day  month  weekday  lag_1  lag_7  \
0              175  83.63        15    3      1        0  100.0  343.0   
1              159  36.30         0    3      1        0   66.0  114.0   
2               85  77.88        15    4      1        1    3.0   22.0   
3              108  45.49         5    4      1        1   58.0   67.0   
4              335  81.01        20    4      1        1   83.0    5.0   

   rolling_mean_7  rolling_mean_14  Units Sold  
0       67.142857       115.714286          66  
1       51.285714       106.857143           3  
2       56.428571       103.571429          58  
3       58.714286        99.000000          83  
4       96.714286        99.000000         271  


In [4]:
#Split feature and target
X = df.drop('Units Sold', axis=1)
Y = df['Units Sold']

In [5]:
#Model Test-Train split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state=42)

In [6]:
#Model 1: Linear Regression
lr = LinearRegression()
lr.fit(X_train, Y_train)

y_pred_lr = lr.predict(X_test)

In [9]:
#Evaluate Linear Regression
mse_lr = np.sqrt(mean_squared_error(Y_test, y_pred_lr))
r2_lr = r2_score(Y_test, y_pred_lr)

print("LinearRegression")
print(f"Mean Squared Error: {mse_lr}")
print(f"R^2 Score: {r2_lr}")

LinearRegression
Mean Squared Error: 83.30189190484067
R^2 Score: 0.41670306414820646


In [10]:
#model 2: Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)

y_pred_rf = rf.predict(X_test)

In [11]:
#Evaluate Random Forest Regressor
mse_rf = np.sqrt(mean_squared_error(Y_test, y_pred_rf))
r2_rf = r2_score(Y_test, y_pred_rf)

print("Random Forest Regressor")
print(f"Mean Squared Error: {mse_rf}")
print(f"R^2 Score: {r2_rf}")

Random Forest Regressor
Mean Squared Error: 81.88187499940662
R^2 Score: 0.43642006455960713


In [12]:
#Model Comparison
result = pd.DataFrame({
    'Model': ['LinearRegression', 'RandomForestRegressor'],
    'RMSE': [mse_lr, mse_rf],
    'R2 Score': [r2_lr, r2_rf]
})

result

,Model,RMSE,R2 Score
0,LinearRegression,83.301892,0.416703
1,RandomForestRegressor,81.881875,0.436420


In [13]:
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance


Inventory Level    0.406414
rolling_mean_7     0.172976
lag_1              0.080234
Price              0.069454
rolling_mean_14    0.068036
lag_7              0.065297
day                0.050502
month              0.036455
weekday            0.028017
Discount           0.022614
dtype: float64

In [14]:
import joblib

joblib.dump(rf, "E:/Inventory_managment/inventory_demand_model.pkl")

['E:/Inventory_managment/inventory_demand_model.pkl']

In [16]:
comparison = pd.DataFrame({
    'Actual Units Sold': Y_test.values[:10],
    'Predicted Units Sold': y_pred_rf[:10]
})

comparison


,Actual Units Sold,Predicted Units Sold
0,17,78.20
1,16,144.26
2,63,106.85
3,60,233.34
4,28,132.07
5,152,113.33
6,138,112.13
7,195,143.99
8,104,154.81
9,25,130.92
